# Unidad 27 — Ordenamiento por burbujas, selección e inserción

Semana 4 · Clase 1 · Fuente: PDF parte 1, págs. 4–67

[Índice de la semana](../semana-04.ipynb) | [Anterior](../semana-04.ipynb) | [Siguiente](u28-ordenamiento-por-fusion.ipynb)

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

## Objetivos de la unidad

Según el PDF (págs. 5–6):

- Comprender el problema de ordenamiento y resolverlo con diversas técnicas.
- Comprender y explicar las diferencias entre el ordenamiento por burbujas, por selección y por inserción.
- Comprender y describir la complejidad temporal de los tres métodos.
- Implementar el ordenamiento de burbujas (intercambia elementos adyacentes), el de selección (encuentra el elemento más pequeño y crea una lista ordenada) y el de inserción (inserta un nuevo elemento en una lista ordenada).

Conceptos previos que el PDF da por sabidos: operadores de comparación, la función de intercambio (swap) y la notación Big O.

Palabras clave: problema de ordenamiento, ordenamiento de burbujas, ordenamiento por selección, ordenamiento por inserción, tiempo cuadrático.

**Referencia:** PDF parte 1, págs. 5–7

## Contexto

Ordenar es una de las tareas más frecuentes en programación: precios de menor a mayor, nombres en orden alfabético, resultados de una competencia. El capítulo 5 del PDF estudia cinco algoritmos de ordenamiento; esta unidad cubre los tres más sencillos (burbuja, selección e inserción) y las unidades 28 y 29 cubren fusión y rápido.

Los tres comparten dos operaciones básicas: **comparar** dos elementos e **intercambiar** sus posiciones. Lo que los distingue es la estrategia con la que eligen qué comparar.

**Referencia:** PDF parte 1, págs. 2–3

## 1. El problema del ordenamiento y la misión de las canicas

El PDF plantea 5 canicas de hierro de igual forma y tamaño pero distinto peso: `50, 30, 40, 10, 20`. Hay que ordenarlas de menor a mayor peso (`10, 20, 30, 40, 50`). Después se plantean tres situaciones, cada una con una herramienta distinta, y cada una corresponde a un algoritmo:

| Situación del PDF | Herramienta | Algoritmo |
|---|---|---|
| Solo se pueden comparar e intercambiar dos canicas **adyacentes**; se envía la más pesada hacia la derecha | dispositivo que no recuerda pesos | Burbuja |
| Se pueden comparar e intercambiar dos canicas **no adyacentes**; se compara la primera con la de menor peso | balanza | Selección |
| Las canicas están en un barril y se sacan **de una en una**; cada una se coloca en una posición ya ordenada | balanza con barril | Inserción |

La pregunta de cada misión es cuántas comparaciones se necesitan; la respuesta general se desarrolla en la sección de complejidad.

**Referencia:** PDF parte 1, págs. 9–12

In [2]:
canicas = [50, 30, 40, 10, 20]

# Python ya trae ordenamiento incorporado: sorted() devuelve una lista nueva y no modifica la original.
print("Original:", canicas)
print("Ordenada:", sorted(canicas))
print("Original sin cambios:", canicas)

Original: [50, 30, 40, 10, 20]
Ordenada: [10, 20, 30, 40, 50]
Original sin cambios: [50, 30, 40, 10, 20]


**Clave:** ordenar significa reubicar los elementos para que cada uno sea menor o igual que el siguiente. Los algoritmos de esta unidad resuelven el problema usando solo comparaciones e intercambios.

**Ampliación (no está en el PDF):** `sorted()` y `list.sort()` usan Timsort, un algoritmo mucho más eficiente que los tres de esta unidad. Aquí se implementan a mano con fines de aprendizaje.

## 2. Operación de intercambio (swap)

Las dos operaciones básicas de un ordenamiento son comparar e intercambiar. El PDF implementa el intercambio de dos maneras.

**Versión con variable temporal (`swap1`):** se guarda `S[x]` en una variable `t`, se copia `S[y]` en `S[x]` y por último `t` en `S[y]`. Sin `t` se perdería uno de los valores.

**Versión con asignación múltiple (`swap2`):** en Python se pueden asignar varios valores a la vez. El lado derecho se convierte en una tupla y luego se asigna al lado izquierdo, así que el intercambio cabe en una línea.

**Referencia:** PDF parte 1, págs. 52–53

In [3]:
def swap1(S, x, y):
    t = S[x]
    S[x] = S[y]
    S[y] = t


S = [10, 20]
print(S)
swap1(S, 0, 1)
print(S)

[10, 20]
[20, 10]


In [4]:
def swap2(S, x, y):
    S[x], S[y] = S[y], S[x]


S = [10, 20]
print(S)
swap2(S, 0, 1)
print(S)

[10, 20]
[20, 10]


**Ampliación (no está en el PDF):** por qué hace falta la variable temporal. Si se escribe el intercambio sin ella, ambos elementos terminan con el mismo valor.

In [5]:
def swap_incorrecto(S, x, y):
    S[x] = S[y]   # aquí se pierde el valor original de S[x]
    S[y] = S[x]   # S[x] ya vale lo mismo que S[y]


S = [10, 20]
swap_incorrecto(S, 0, 1)
print(S)  # [20, 20]: se perdió el 10

[20, 20]


**Clave:** `S[x], S[y] = S[y], S[x]` es la forma idiomática de Python y es la que usan todos los algoritmos siguientes.

**Error común:** intercambiar con dos asignaciones consecutivas sin variable temporal; se pierde un valor.

**Para el quiz:** el intercambio por asignación múltiple funciona porque el lado derecho se evalúa completo (como tupla) antes de asignar.

**Referencia:** PDF parte 1, págs. 52–53

## 3. Ordenamiento por burbujas

### 3.1 Idea y proceso paso a paso

El ordenamiento por burbujas compara dos elementos **adyacentes** e intercambia sus posiciones si están en orden invertido. El PDF lo ilustra con `[50, 30, 40, 10, 20]`:

**Primera pasada**

1. Se compara 50 con 30: 50 es mayor, se intercambian: `[30, 50, 40, 10, 20]`.
2. Se compara 50 con 40: se intercambian: `[30, 40, 50, 10, 20]`.
3. Se compara 50 con 10: se intercambian: `[30, 40, 10, 50, 20]`.
4. Se compara 50 con 20: se intercambian: `[30, 40, 10, 20, 50]`.

Al terminar la primera pasada el elemento más grande (50) queda en el extremo derecho, ya ordenado, y puede excluirse de la segunda pasada.

**Segunda pasada** (sobre `[30, 40, 10, 20]`): 30 y 40 no se intercambian; 40 con 10 se intercambian; 40 con 20 se intercambian. Resultado: `[30, 10, 20, 40, 50]`. Ya hay dos elementos ordenados al final.

**Tercera pasada**: 30 con 10 se intercambian; 30 con 20 se intercambian. Resultado: `[10, 20, 30, 40, 50]`.

**Cuarta pasada**: 10 con 20 no se intercambian; la lista queda ordenada.

**Referencia:** PDF parte 1, págs. 14–23

In [6]:
# Ampliación (no está en el PDF): traza que reproduce los diagramas de las págs. 15-23.
def traza_burbuja(lista):
    S = list(lista)
    n = len(S)
    for i in range(n - 1):
        print(f"Pasada {i + 1}: se revisan las posiciones 0 a {n - 2 - i}")
        for j in range(n - 1 - i):
            if S[j] > S[j + 1]:
                S[j], S[j + 1] = S[j + 1], S[j]
                print(f"  compara {S[j + 1]} > {S[j]}: intercambia -> {S}")
            else:
                print(f"  compara {S[j]} <= {S[j + 1]}: no intercambia -> {S}")
    return S


traza_burbuja([50, 30, 40, 10, 20])

Pasada 1: se revisan las posiciones 0 a 3
  compara 50 > 30: intercambia -> [30, 50, 40, 10, 20]
  compara 50 > 40: intercambia -> [30, 40, 50, 10, 20]
  compara 50 > 10: intercambia -> [30, 40, 10, 50, 20]
  compara 50 > 20: intercambia -> [30, 40, 10, 20, 50]
Pasada 2: se revisan las posiciones 0 a 2
  compara 30 <= 40: no intercambia -> [30, 40, 10, 20, 50]
  compara 40 > 10: intercambia -> [30, 10, 40, 20, 50]
  compara 40 > 20: intercambia -> [30, 10, 20, 40, 50]
Pasada 3: se revisan las posiciones 0 a 1
  compara 30 > 10: intercambia -> [10, 30, 20, 40, 50]
  compara 30 > 20: intercambia -> [10, 20, 30, 40, 50]
Pasada 4: se revisan las posiciones 0 a 0
  compara 10 <= 20: no intercambia -> [10, 20, 30, 40, 50]


[10, 20, 30, 40, 50]

**Ampliación (no está en el PDF):** en la traza la cuarta pasada (solo compara 10 con 20) no intercambia nada. Esa observación lleva a la optimización de la sección 3.3.

### 3.2 Implementación del PDF

El PDF implementa el algoritmo con un doble bucle `for` y muestra el estado de `S` al comienzo de cada pasada del bucle externo (`i`).

In [7]:
def bubblesort(S):
    n = len(S)
    for i in range(n):
        print(S)
        for j in range(n - 1):
            if S[j] > S[j + 1]:
                S[j], S[j + 1] = S[j + 1], S[j]


S = [50, 30, 40, 10, 20]
bubblesort(S)
print(S)

[50, 30, 40, 10, 20]
[30, 40, 10, 20, 50]
[30, 10, 20, 40, 50]
[10, 20, 30, 40, 50]
[10, 20, 30, 40, 50]
[10, 20, 30, 40, 50]


**Nota sobre el PDF:** el código de la pág. 54 siempre recorre `n - 1` comparaciones en cada pasada, aunque los últimos elementos ya estén ordenados; es decir, no aplica la exclusión de la parte ordenada que describen los diagramas. Funciona igual, pero hace más comparaciones (`N(N-1)` en total, no `N(N-1)/2`). La pág. 59 calcula `N(N-1)/2`, que corresponde a la versión que sí excluye la parte ordenada. La salida impresa coincide con la del PDF: 5 líneas del estado al inicio de cada pasada más la impresión final.

### 3.3 Versión optimizada

**Ampliación (no está en el PDF):** dos mejoras clásicas. La primera reduce el recorrido (`n - 1 - i`) porque tras `i` pasadas los últimos `i` elementos están en su lugar. La segunda usa una bandera: si una pasada completa no intercambia nada, la lista ya está ordenada y se puede terminar.

In [8]:
def bubblesort_optimizado(S):
    n = len(S)
    for i in range(n - 1):
        hubo_intercambio = False
        for j in range(n - 1 - i):
            if S[j] > S[j + 1]:
                S[j], S[j + 1] = S[j + 1], S[j]
                hubo_intercambio = True
        if not hubo_intercambio:
            break   # ninguna pareja estaba invertida: ya está ordenada


S = [50, 30, 40, 10, 20]
bubblesort_optimizado(S)
print(S)

ya_ordenada = [10, 20, 30, 40, 50]
bubblesort_optimizado(ya_ordenada)  # termina tras una sola pasada
print(ya_ordenada)

[10, 20, 30, 40, 50]
[10, 20, 30, 40, 50]


**Clave:** en cada pasada el mayor de los elementos no ordenados "sube como una burbuja" hasta el final. Se necesitan como máximo `N - 1` pasadas.

**Error común:** usar `range(n)` en el bucle interno y acceder a `S[j + 1]`: provoca `IndexError` en la última comparación. Por eso el límite interno es `n - 1`.

**Para el quiz:** burbuja solo compara elementos **adyacentes** y el intercambio ocurre únicamente si el primero es mayor que el segundo.

**Referencia:** PDF parte 1, págs. 14–23 (proceso) y 54 (código)

## 4. Ordenamiento por selección

### 4.1 Idea y proceso paso a paso

El ordenamiento por selección encuentra el elemento más pequeño de la lista y lo coloca a la izquierda. La estrategia que muestra el PDF compara el **primer** elemento de la parte sin ordenar con cada uno de los siguientes y los intercambia cuando el segundo es menor. Con `[50, 30, 40, 10, 20]`:

**Primera pasada** (posición 0):

1. 50 contra 30: 50 es mayor, se intercambian: `[30, 50, 40, 10, 20]`.
2. 30 contra 40: no se intercambian.
3. 30 contra 10: se intercambian: `[10, 50, 40, 30, 20]`.
4. 10 contra 20: no se intercambian.

El elemento más pequeño (10) queda al inicio y se excluye de las siguientes pasadas.

**Segunda pasada** (posición 1, sobre `[50, 40, 30, 20]`): 50 con 40 se intercambian; 40 con 30 se intercambian; 30 con 20 se intercambian: `[10, 20, 50, 40, 30]`.

**Tercera pasada**: 50 con 40 se intercambian; 40 con 30 se intercambian: `[10, 20, 30, 50, 40]`.

**Cuarta pasada**: 50 con 40 se intercambian: `[10, 20, 30, 40, 50]`.

**Referencia:** PDF parte 1, págs. 24–36

In [9]:
# Ampliación (no está en el PDF): la estrategia de los diagramas (intercambiar cada vez que se encuentra uno menor).
def traza_seleccion_por_intercambios(lista):
    S = list(lista)
    n = len(S)
    for i in range(n - 1):
        print(f"Pasada {i + 1}: posición {i}")
        for j in range(i + 1, n):
            if S[i] > S[j]:
                print(f"  compara {S[i]} con {S[j]}: intercambia", end="")
                S[i], S[j] = S[j], S[i]
                print(f" -> {S}")
            else:
                print(f"  compara {S[i]} con {S[j]}: no intercambia -> {S}")
    return S


traza_seleccion_por_intercambios([50, 30, 40, 10, 20])

Pasada 1: posición 0
  compara 50 con 30: intercambia -> [30, 50, 40, 10, 20]
  compara 30 con 40: no intercambia -> [30, 50, 40, 10, 20]
  compara 30 con 10: intercambia -> [10, 50, 40, 30, 20]
  compara 10 con 20: no intercambia -> [10, 50, 40, 30, 20]
Pasada 2: posición 1
  compara 50 con 40: intercambia -> [10, 40, 50, 30, 20]
  compara 40 con 30: intercambia -> [10, 30, 50, 40, 20]
  compara 30 con 20: intercambia -> [10, 20, 50, 40, 30]
Pasada 3: posición 2
  compara 50 con 40: intercambia -> [10, 20, 40, 50, 30]
  compara 40 con 30: intercambia -> [10, 20, 30, 50, 40]
Pasada 4: posición 3
  compara 50 con 40: intercambia -> [10, 20, 30, 40, 50]


[10, 20, 30, 40, 50]

### 4.2 Implementación sin intercambio, con lista extra (`selectionsort1`)

Esta versión toma la lista no ordenada `S` y construye la lista ordenada `R`: busca el elemento más pequeño de `S`, lo agrega a `R` y lo elimina de `S`; repite hasta que `S` queda vacía. El código imprime `R` y `S` en cada paso.

**Referencia:** PDF parte 1, pág. 55

In [10]:
def selectionsort1(S):
    R = []
    while len(S) > 0:
        print(R, S)
        smallest = S.index(min(S))
        R.append(S[smallest])
        S.pop(smallest)
    return R


S = [50, 30, 40, 10, 20]
R = selectionsort1(S)
print(R)
print("S después de ordenar:", S)

[] [50, 30, 40, 10, 20]
[10] [50, 30, 40, 20]
[10, 20] [50, 30, 40]
[10, 20, 30] [50, 40]
[10, 20, 30, 40] [50]
[10, 20, 30, 40, 50]
S después de ordenar: []


**Nota sobre el PDF:** el PDF no lo menciona, pero `selectionsort1` **vacía** la lista original (`S.pop`). Tras la llamada `S` es `[]`; el resultado está solo en `R`. Si se necesita conservar la original hay que pasar una copia: `selectionsort1(list(S))`.

### 4.3 Implementación in situ (`selectionsort2`)

Esta versión ordena directamente en la misma lista, sin memoria adicional. Por cada posición `i` busca el índice del menor valor entre las posiciones restantes y hace **un solo** intercambio con la posición `i`.

**Referencia:** PDF parte 1, pág. 56

In [11]:
def selectionsort2(S):
    n = len(S)
    for i in range(n - 1):
        print(S)
        smallest = i
        for j in range(i + 1, n):
            if S[j] < S[smallest]:
                smallest = j
        S[i], S[smallest] = S[smallest], S[i]


S = [50, 30, 40, 10, 20]
selectionsort2(S)
print(S)

[50, 30, 40, 10, 20]
[10, 30, 40, 50, 20]
[10, 20, 40, 50, 30]
[10, 20, 30, 50, 40]
[10, 20, 30, 40, 50]


**Nota sobre el PDF:** los diagramas (págs. 25–36) intercambian cada vez que encuentran un elemento menor, mientras que el código `selectionsort2` (pág. 56) recuerda **la posición del menor** y hace un único intercambio por pasada. Por eso los estados intermedios no coinciden (por ejemplo, tras la primera pasada el diagrama muestra `[10, 50, 40, 30, 20]` y el código `[10, 30, 40, 50, 20]`), aunque el resultado final es el mismo. El código es la variante habitual: hace como máximo `N - 1` intercambios.

**Clave:** selección siempre hace `N(N-1)/2` comparaciones, sin importar cómo vengan los datos; en cambio hace pocos intercambios.

**Error común:** confundir "el menor" con "su posición". Hay que guardar el **índice** (`smallest = j`), no el valor, para poder intercambiar después.

**Para el quiz:** burbuja y selección tienen la misma complejidad `O(N²)` porque ambos usan un doble bucle `for`.

**Referencia:** PDF parte 1, págs. 24–36 (proceso) y 55–56 (código)

## 5. Ordenamiento por inserción

### 5.1 Idea y proceso paso a paso

El ordenamiento por inserción añade, uno a uno, los elementos de la parte no ordenada a la parte ordenada. Para cada elemento nuevo busca su posición comparándolo secuencialmente (de derecha a izquierda) con la parte ordenada. Con `[50, 30, 40, 10, 20]`:

1. El primer elemento (50) ya es una lista ordenada de un solo elemento.
2. Se toma 30: es menor que 50, así que 50 se desplaza una posición a la derecha. No quedan más elementos que comparar, se inserta 30: `[30, 50, 40, 10, 20]`.
3. Se toma 40: 50 se desplaza a la derecha; el anterior (30) es menor que 40, se inserta 40: `[30, 40, 50, 10, 20]`.
4. Se toma 10: se desplazan 50, 40 y 30; no quedan más elementos, se inserta 10: `[10, 30, 40, 50, 20]`.
5. Se toma 20: se desplazan 50, 40 y 30; el anterior (10) es menor que 20, se inserta 20: `[10, 20, 30, 40, 50]`.

**Referencia:** PDF parte 1, págs. 37–50

In [12]:
# Ampliación (no está en el PDF): traza que muestra los desplazamientos de las págs. 38-50.
def traza_insercion(lista):
    S = list(lista)
    for i in range(1, len(S)):
        x = S[i]
        j = i - 1
        print(f"Se toma {x} (posición {i}); parte ordenada: {S[:i]}")
        while j >= 0 and S[j] > x:
            print(f"  {S[j]} > {x}: se desplaza {S[j]} a la derecha")
            S[j + 1] = S[j]
            j -= 1
        S[j + 1] = x
        print(f"  se inserta {x} en la posición {j + 1} -> {S}")
    return S


traza_insercion([50, 30, 40, 10, 20])

Se toma 30 (posición 1); parte ordenada: [50]
  50 > 30: se desplaza 50 a la derecha
  se inserta 30 en la posición 0 -> [30, 50, 40, 10, 20]
Se toma 40 (posición 2); parte ordenada: [30, 50]
  50 > 40: se desplaza 50 a la derecha
  se inserta 40 en la posición 1 -> [30, 40, 50, 10, 20]
Se toma 10 (posición 3); parte ordenada: [30, 40, 50]
  50 > 10: se desplaza 50 a la derecha
  40 > 10: se desplaza 40 a la derecha
  30 > 10: se desplaza 30 a la derecha
  se inserta 10 en la posición 0 -> [10, 30, 40, 50, 20]
Se toma 20 (posición 4); parte ordenada: [10, 30, 40, 50]
  50 > 20: se desplaza 50 a la derecha
  40 > 20: se desplaza 40 a la derecha
  30 > 20: se desplaza 30 a la derecha
  se inserta 20 en la posición 1 -> [10, 20, 30, 40, 50]


[10, 20, 30, 40, 50]

### 5.2 Con memoria adicional (`insertionsort1`)

Toma la lista desordenada `S` y construye la lista ordenada `R`: extrae el primer elemento de `S`, busca en `R` (recorriéndola en orden inverso) la posición donde debe insertarse y lo inserta con `R.insert`. Imprime `R` y `S` en cada paso.

**Referencia:** PDF parte 1, pág. 57

In [13]:
def insertionsort1(S):
    n = len(S)
    R = []
    while len(S) > 0:
        print(R, S)
        x = S.pop(0)
        j = len(R) - 1
        while j >= 0 and R[j] > x:
            j -= 1
        R.insert(j + 1, x)
    return R


S = [50, 30, 40, 10, 20]
R = insertionsort1(S)
print(R)

[] [50, 30, 40, 10, 20]
[50] [30, 40, 10, 20]
[30, 50] [40, 10, 20]
[30, 40, 50] [10, 20]
[10, 30, 40, 50] [20]
[10, 20, 30, 40, 50]


**Nota sobre el PDF:** el listado de la diapositiva incluye `n = len(S)` (línea 2), variable que nunca se usa; se conserva por fidelidad, pero puede eliminarse. Como `selectionsort1`, esta función también vacía la lista original `S`.

### 5.3 In situ (`insertionsort2`)

Ordena la lista directamente: el bucle externo empieza en la segunda posición; en cada paso guarda el valor actual `x`, desplaza a la derecha los elementos mayores que `x` y por último inserta `x` en su lugar.

**Referencia:** PDF parte 1, pág. 58

In [14]:
def insertionsort2(S):
    n = len(S)
    for i in range(1, n):
        print(S)
        x = S[i]
        j = i - 1
        while j >= 0 and S[j] > x:
            S[j + 1] = S[j]
            j -= 1
        S[j + 1] = x


S = [50, 30, 40, 10, 20]
insertionsort2(S)
print(S)

[50, 30, 40, 10, 20]
[30, 50, 40, 10, 20]
[30, 40, 50, 10, 20]
[10, 30, 40, 50, 20]
[10, 20, 30, 40, 50]


**Nota sobre el PDF:** la salida mostrada en la pág. 58 tiene 4 líneas y empieza en `[30, 50, 40, 10, 20]`. Con el código del propio PDF se obtienen 4 impresiones del bucle (la primera es la lista original `[50, 30, 40, 10, 20]`) más la impresión final: 5 líneas en total. La diapositiva muestra una línea menos (no aparece la lista original).

**Clave:** la condición `j >= 0 and S[j] > x` se evalúa de izquierda a derecha con cortocircuito; si `j` fuera negativo no se llega a evaluar `S[j]`.

**Error común:** escribir la condición al revés (`S[j] > x and j >= 0`): cuando `j` vale `-1`, `S[-1]` es el último elemento de la lista y se compara con un valor equivocado, sin lanzar error.

**Para el quiz:** inserción es el único de los tres cuyo mejor caso (lista ya ordenada) es `O(N)`.

**Referencia:** PDF parte 1, págs. 37–50 (proceso) y 57–58 (código)

## 6. Complejidad temporal y comparación entre los tres

### 6.1 Lo que dice el PDF

- **Burbuja:** doble bucle `for` anidado; el número de comparaciones por pasada es `N-1, N-2, ..., 1`. Total: `T(N) = (N-1) + (N-2) + ... + 1 = N(N-1)/2`, por lo tanto `O(N²)` (tiempo cuadrático).
- **Selección:** doble bucle como burbuja, por lo tanto también `O(N²)`.
- **Inserción:** el número de comparaciones depende del caso.
  - Mejor caso: lista ya ordenada. Basta una comparación por pasada, así que es `O(N)`.
  - Peor caso: lista en orden inverso. Cada elemento se compara con todos los de la parte ordenada, por lo tanto `O(N²)`.

**Referencia:** PDF parte 1, pág. 59

### 6.2 Medirlo en lugar de creerlo

**Ampliación (no está en el PDF):** se instrumentan las tres versiones para contar comparaciones e intercambios (o desplazamientos) y comprobar las fórmulas. Las versiones de burbuja y selección aquí son las que excluyen la parte ordenada.

In [15]:
def burbuja_contada(lista):
    """Burbuja que excluye la parte ordenada. Devuelve (ordenada, comparaciones, intercambios)."""
    S = list(lista)
    n = len(S)
    comparaciones = intercambios = 0
    for i in range(n - 1):
        for j in range(n - 1 - i):
            comparaciones += 1
            if S[j] > S[j + 1]:
                S[j], S[j + 1] = S[j + 1], S[j]
                intercambios += 1
    return S, comparaciones, intercambios


def seleccion_contada(lista):
    S = list(lista)
    n = len(S)
    comparaciones = intercambios = 0
    for i in range(n - 1):
        menor = i
        for j in range(i + 1, n):
            comparaciones += 1
            if S[j] < S[menor]:
                menor = j
        S[i], S[menor] = S[menor], S[i]
        intercambios += 1
    return S, comparaciones, intercambios


def insercion_contada(lista):
    """Devuelve (ordenada, comparaciones, desplazamientos)."""
    S = list(lista)
    comparaciones = desplazamientos = 0
    for i in range(1, len(S)):
        x = S[i]
        j = i - 1
        while j >= 0:
            comparaciones += 1          # se evalúa S[j] > x
            if S[j] > x:
                S[j + 1] = S[j]
                desplazamientos += 1
                j -= 1
            else:
                break
        S[j + 1] = x
    return S, comparaciones, desplazamientos


import random

random.seed(42)   # semilla fija: los datos son siempre los mismos
N = 10
casos = {
    "ordenada": list(range(N)),
    "inversa": list(range(N, 0, -1)),
    "aleatoria": random.sample(range(100), N),
}
print(f"N = {N}; N(N-1)/2 = {N * (N - 1) // 2}")
print(f"{'caso':<10}{'algoritmo':<11}{'comparaciones':>14}{'intercambios/desplaz.':>23}")
for nombre, datos in casos.items():
    for alg, f in (("burbuja", burbuja_contada), ("seleccion", seleccion_contada), ("insercion", insercion_contada)):
        _, c, m = f(datos)
        print(f"{nombre:<10}{alg:<11}{c:>14}{m:>23}")

N = 10; N(N-1)/2 = 45
caso      algoritmo   comparaciones  intercambios/desplaz.
ordenada  burbuja                45                      0
ordenada  seleccion              45                      9
ordenada  insercion               9                      0
inversa   burbuja                45                     45
inversa   seleccion              45                      9
inversa   insercion              45                     45
aleatoria burbuja                45                     25
aleatoria seleccion              45                      9
aleatoria insercion              32                     25


Se observa en la tabla:

- Burbuja y selección hacen siempre `N(N-1)/2 = 45` comparaciones, sin importar el orden de los datos.
- Inserción hace solo `N - 1 = 9` comparaciones con la lista ordenada (mejor caso, `O(N)`) y `45` con la lista inversa (peor caso, `O(N²)`).
- Selección hace como máximo `N - 1` intercambios; burbuja puede hacer `N(N-1)/2` en el peor caso.

### 6.3 Mediciones de tiempo

**Ampliación (no está en el PDF):** al duplicar `N`, un algoritmo cuadrático tarda unas 4 veces más. Los tiempos dependen de la máquina y de la carga del momento, por lo que son **orientativos**: lo que importa es la proporción, no los segundos.

In [16]:
import random
import time

random.seed(42)
print("Mediciones orientativas (dependen de la máquina)")
print(f"{'N':>6}{'burbuja':>12}{'seleccion':>12}{'insercion':>12}   (segundos)")
for N in (250, 500, 1000):
    datos = random.sample(range(10 * N), N)
    fila = []
    for f in (burbuja_contada, seleccion_contada, insercion_contada):
        t0 = time.perf_counter()
        f(datos)
        fila.append(time.perf_counter() - t0)
    print(f"{N:>6}{fila[0]:>12.4f}{fila[1]:>12.4f}{fila[2]:>12.4f}")

Mediciones orientativas (dependen de la máquina)
     N     burbuja   seleccion   insercion   (segundos)
   250      0.0011      0.0005      0.0005
   500      0.0044      0.0024      0.0021
  1000      0.0186      0.0095      0.0088


### 6.4 Resumen comparativo

**Ampliación (no está en el PDF)** salvo las complejidades, que sí figuran en la pág. 59:

| | Burbuja | Selección | Inserción |
|---|---|---|---|
| Estrategia | intercambiar adyacentes invertidos | elegir el menor y llevarlo a su posición | insertar cada elemento en la parte ordenada |
| Mejor caso | `O(N²)` (con la versión del PDF); `O(N)` con bandera | `O(N²)` | `O(N)` |
| Peor caso | `O(N²)` | `O(N²)` | `O(N²)` |
| Intercambios | hasta `N(N-1)/2` | como máximo `N-1` | no intercambia: desplaza |
| In situ | sí | sí (versión 2) | sí (versión 2) |
| Estable | sí | no (versión con intercambio) | sí |

Un algoritmo es **estable** si no altera el orden relativo de elementos iguales. Se retoma en el ejercicio 8.

**Clave:** los tres son `O(N²)` en el peor caso y por eso sirven para listas pequeñas o casi ordenadas; para listas grandes se usan algoritmos como fusión y rápido (unidades 28 y 29).

**Para el quiz:** inserción con lista ordenada: `O(N)`; con lista inversa: `O(N²)`.

**Referencia:** PDF parte 1, pág. 59

## 7. Prueba sorpresa del PDF

**Q1 (pág. 61):** ¿cuántas operaciones de intercambio se ejecutan en el ordenamiento de burbujas de `[50, 30, 40, 10, 20]`? (el PDF resalta la línea del intercambio).

**Q2 (pág. 62):** ¿cuántas operaciones de comparación se ejecutan en el ordenamiento por inserción de la misma lista? (el PDF resalta `S[j] > x`).

El PDF no incluye las respuestas; se calculan contando dentro del propio código.

**Referencia:** PDF parte 1, págs. 61–62

In [17]:
def bubblesort_contando(S):
    intercambios = 0
    n = len(S)
    for i in range(n):
        for j in range(n - 1):
            if S[j] > S[j + 1]:
                S[j], S[j + 1] = S[j + 1], S[j]
                intercambios += 1   # línea 7 resaltada en el PDF
    return intercambios


def insertionsort2_contando(S):
    comparaciones = 0
    for i in range(1, len(S)):
        x = S[i]
        j = i - 1
        while j >= 0:
            comparaciones += 1      # evaluación de S[j] > x (con j >= 0 verdadero)
            if not S[j] > x:
                break
            S[j + 1] = S[j]
            j -= 1
        S[j + 1] = x
    return comparaciones


print("Q1 intercambios en burbuja:", bubblesort_contando([50, 30, 40, 10, 20]))
print("Q2 comparaciones S[j] > x en inserción:", insertionsort2_contando([50, 30, 40, 10, 20]))

Q1 intercambios en burbuja: 8
Q2 comparaciones S[j] > x en inserción: 10


- **Q1: 8 intercambios.** Coincide con el número de **inversiones** de la lista (parejas fuera de orden): 50 tiene 4 menores a su derecha, 30 tiene 2 (10 y 20) y 40 tiene 2.
- **Q2: 10 comparaciones.** Por elemento insertado: 30 → 1; 40 → 2; 10 → 3; 20 → 4. Cuando `j` llega a `-1`, el cortocircuito evita evaluar `S[j] > x`, así que ese caso no cuenta.

**Clave:** los intercambios de burbuja equivalen a las inversiones de la lista; los desplazamientos de inserción también.

## Proyecto de la unidad

El PDF no trae un proyecto con `input()`; cierra la unidad con una sección de **programación en pareja** (págs. 63–67). Se resuelve aquí como proyecto de la unidad.

### Práctica de programación en pareja (resumen de las págs. 64–65)

- Preparar lineamientos y mecanismos: los alumnos se turnan para manejar el ratón (el "conductor") y se prevé un plan de contingencia si un integrante falta.
- Emparejar habilidades similares, no necesariamente iguales; la programación en pareja no es "dividir y conquistar", sino un esfuerzo colaborativo en todo el proyecto.
- Ofrecer incentivos adicionales y exigir la pareja solo en una o dos tareas.
- Evitar trampas: evaluar resultados individuales, explicar el valor de lo que se aprende y discutir de forma explícita qué conductas se consideran trampa.
- Un entorno de aprendizaje colaborativo implica que los estudiantes trabajen juntos en actividades de aprendizaje, con conversación intelectual entre ellos.

### Problema (págs. 66–67)

**Q1.** Dadas dos palabras, escribir un algoritmo que determine si son **anagramas**: una palabra que se forma reordenando todas las letras de otra, usando cada letra exactamente una vez. El PDF usa `LISTEN` y `SILENT` en el diagrama (en la traducción aparece como "ESCUCHAR" y "SILENCIO", que en español **no** son anagramas: tienen distinta cantidad de letras).

**Q2.** Usar ordenamiento para facilitarlo:

- Una función que evalúe anagramas con `sorted()`.
- Modificar `selectionsort2()` para determinarlo.
- Modificar `insertionsort2()` para determinarlo.

**Nota sobre el PDF:** el ejemplo en español "ESCUCHAR" y "SILENCIO" es un error de traducción automática de `LISTEN` y `SILENT`. Se usan en el código los pares `listen`/`silent` y un par en español, `amor`/`roma`.

### Plan de programación

1. Normalizar las palabras (minúsculas) y descartar de inmediato si tienen distinta longitud.
2. Convertir cada palabra en lista de letras y ordenarla.
3. Dos palabras son anagramas si, una vez ordenadas, las listas son iguales.
4. Repetir el paso 2 con tres técnicas: `sorted()`, selección in situ e inserción in situ.

**Referencia:** PDF parte 1, págs. 64–67

In [18]:
def es_anagrama_sorted(a, b):
    """Anagrama usando sorted() (pág. 67, primera viñeta)."""
    a, b = a.lower(), b.lower()
    return len(a) == len(b) and sorted(a) == sorted(b)


def selection_sort_letras(S):
    """selectionsort2 sin impresiones: ordena la lista de letras in situ."""
    n = len(S)
    for i in range(n - 1):
        smallest = i
        for j in range(i + 1, n):
            if S[j] < S[smallest]:
                smallest = j
        S[i], S[smallest] = S[smallest], S[i]


def es_anagrama_seleccion(a, b):
    """Anagrama modificando selectionsort2 (pág. 67, segunda viñeta)."""
    a, b = a.lower(), b.lower()
    if len(a) != len(b):
        return False
    la, lb = list(a), list(b)
    selection_sort_letras(la)
    selection_sort_letras(lb)
    return la == lb


def insertion_sort_letras(S):
    """insertionsort2 sin impresiones: ordena la lista de letras in situ."""
    for i in range(1, len(S)):
        x = S[i]
        j = i - 1
        while j >= 0 and S[j] > x:
            S[j + 1] = S[j]
            j -= 1
        S[j + 1] = x


def es_anagrama_insercion(a, b):
    """Anagrama modificando insertionsort2 (pág. 67, tercera viñeta)."""
    a, b = a.lower(), b.lower()
    if len(a) != len(b):
        return False
    la, lb = list(a), list(b)
    insertion_sort_letras(la)
    insertion_sort_letras(lb)
    return la == lb


pares = [("listen", "silent"), ("amor", "roma"), ("hello", "world"), ("abc", "abcd"), ("Python", "typhon")]
for a, b in pares:
    r = (es_anagrama_sorted(a, b), es_anagrama_seleccion(a, b), es_anagrama_insercion(a, b))
    print(f"{a!r} y {b!r}: sorted={r[0]}, selección={r[1]}, inserción={r[2]}")

'listen' y 'silent': sorted=True, selección=True, inserción=True
'amor' y 'roma': sorted=True, selección=True, inserción=True
'hello' y 'world': sorted=False, selección=False, inserción=False
'abc' y 'abcd': sorted=False, selección=False, inserción=False
'Python' y 'typhon': sorted=True, selección=True, inserción=True


**Resultado esperado:** los tres métodos coinciden en todos los pares: `listen`/`silent`, `amor`/`roma` y `Python`/`typhon` son anagramas; `hello`/`world` y `abc`/`abcd` no lo son.

### Versión interactiva y versión de prueba

La función `anagramas_interactivo` pide pares de palabras hasta que el usuario cancela (entrada vacía o `None`). Recibe la función de lectura como parámetro: por defecto usa `input`, y para la prueba no interactiva se le pasa un lector que entrega una lista de entradas de prueba.

In [19]:
def anagramas_interactivo(leer=input):
    """Pide pares de palabras y dice si son anagramas. Termina con entrada vacía o None."""
    while True:
        a = leer("Primera palabra (vacío para salir): ")
        if a is None or a.strip() == "":
            print("Fin del programa.")
            break
        b = leer("Segunda palabra (vacío para salir): ")
        if b is None or b.strip() == "":
            print("Fin del programa.")
            break
        if es_anagrama_sorted(a, b):
            print(f"{a!r} y {b!r} son anagramas.")
        else:
            print(f"{a!r} y {b!r} no son anagramas.")


def lector_de_prueba(valores):
    """Simula input() entregando valores en orden; devuelve None cuando se agotan (cancelación)."""
    iterador = iter(valores)

    def leer(mensaje=""):
        valor = next(iterador, None)
        print(f"{mensaje}{valor if valor is not None else '<cancelado>'}")
        return valor

    return leer


# Versión no interactiva: se ejecuta con entradas de prueba (la última cadena vacía cancela).
anagramas_interactivo(lector_de_prueba(["listen", "silent", "amor", "roma", "hello", "world", ""]))

# Para usarlo de forma interactiva, quita el comentario de la siguiente línea
# anagramas_interactivo()

Primera palabra (vacío para salir): listen
Segunda palabra (vacío para salir): silent
'listen' y 'silent' son anagramas.
Primera palabra (vacío para salir): amor
Segunda palabra (vacío para salir): roma
'amor' y 'roma' son anagramas.
Primera palabra (vacío para salir): hello
Segunda palabra (vacío para salir): world
'hello' y 'world' no son anagramas.
Primera palabra (vacío para salir): 
Fin del programa.


## Práctica

Ocho ejercicios de dificultad creciente. Completa cada función y ejecuta la celda de verificación; hasta que no la resuelvas imprimirá "Incorrecto: ...", lo cual es normal.

### Ejercicio 1 — Intercambiar dos posiciones

Escribe `intercambiar(S, x, y)` que intercambie en la misma lista los elementos de las posiciones `x` e `y` y **devuelva** la lista. Usa asignación múltiple.

In [20]:
def intercambiar(S, x, y):
    # Escribe tu solución aquí
    pass

In [21]:
verificar("intercambia 0 y 1", intercambiar([10, 20], 0, 1), [20, 10])
verificar("intercambia 0 y 3", intercambiar([1, 2, 3, 4], 0, 3), [4, 2, 3, 1])
verificar("misma posición no cambia nada", intercambiar([5, 6, 7], 1, 1), [5, 6, 7])

Incorrecto: intercambia 0 y 1. Se esperaba [20, 10] y se obtuvo None
Incorrecto: intercambia 0 y 3. Se esperaba [4, 2, 3, 1] y se obtuvo None
Incorrecto: misma posición no cambia nada. Se esperaba [5, 6, 7] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def intercambiar(S, x, y):
    S[x], S[y] = S[y], S[x]
    return S
```

La asignación múltiple evalúa primero la tupla `(S[y], S[x])` y luego la asigna, así que no hace falta variable temporal.
</details>

### Ejercicio 2 — Contar los intercambios de burbuja

Escribe `intercambios_burbuja(lista)` que ordene una **copia** de la lista con burbuja y devuelva cuántos intercambios hizo (es la pregunta Q1 de la prueba sorpresa).

In [22]:
def intercambios_burbuja(lista):
    # Escribe tu solución aquí
    pass

In [23]:
verificar("lista del PDF", intercambios_burbuja([50, 30, 40, 10, 20]), 8)
verificar("ya ordenada", intercambios_burbuja([1, 2, 3]), 0)
verificar("inversa de 4", intercambios_burbuja([4, 3, 2, 1]), 6)

Incorrecto: lista del PDF. Se esperaba 8 y se obtuvo None
Incorrecto: ya ordenada. Se esperaba 0 y se obtuvo None
Incorrecto: inversa de 4. Se esperaba 6 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def intercambios_burbuja(lista):
    S = list(lista)
    cuenta = 0
    n = len(S)
    for i in range(n - 1):
        for j in range(n - 1 - i):
            if S[j] > S[j + 1]:
                S[j], S[j + 1] = S[j + 1], S[j]
                cuenta += 1
    return cuenta
```

Se trabaja sobre una copia (`list(lista)`) para no modificar la original. El resultado es el número de inversiones: para `[4, 3, 2, 1]` son 6.
</details>

### Ejercicio 3 — Burbuja descendente

Escribe `burbuja_descendente(lista)` que devuelva una lista nueva ordenada de **mayor a menor** con burbuja, sin modificar la original.

In [24]:
def burbuja_descendente(lista):
    # Escribe tu solución aquí
    pass

In [25]:
verificar("lista del PDF", burbuja_descendente([50, 30, 40, 10, 20]), [50, 40, 30, 20, 10])
verificar("con repetidos", burbuja_descendente([2, 3, 2, 1]), [3, 2, 2, 1])
original = [3, 1, 2]
burbuja_descendente(original)
verificar("no modifica la original", original, [3, 1, 2])

Incorrecto: lista del PDF. Se esperaba [50, 40, 30, 20, 10] y se obtuvo None
Incorrecto: con repetidos. Se esperaba [3, 2, 2, 1] y se obtuvo None
Correcto: no modifica la original


<details><summary>Ver solución</summary>

```python
def burbuja_descendente(lista):
    S = list(lista)
    n = len(S)
    for i in range(n - 1):
        for j in range(n - 1 - i):
            if S[j] < S[j + 1]:
                S[j], S[j + 1] = S[j + 1], S[j]
    return S
```

Basta invertir la comparación: se intercambia cuando el primero es **menor** que el segundo.
</details>

### Ejercicio 4 — Posición del mínimo

Escribe `posicion_minimo(S, desde)` que devuelva el **índice** del menor valor de `S[desde:]`. Es el paso central del ordenamiento por selección. Si hay empate, devuelve el primero.

In [26]:
def posicion_minimo(S, desde):
    # Escribe tu solución aquí
    pass

In [27]:
verificar("desde 0", posicion_minimo([50, 30, 40, 10, 20], 0), 3)
verificar("desde 4", posicion_minimo([50, 30, 40, 10, 20], 4), 4)
verificar("empate: el primero", posicion_minimo([5, 2, 9, 2], 0), 1)

Incorrecto: desde 0. Se esperaba 3 y se obtuvo None
Incorrecto: desde 4. Se esperaba 4 y se obtuvo None
Incorrecto: empate: el primero. Se esperaba 1 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def posicion_minimo(S, desde):
    menor = desde
    for j in range(desde + 1, len(S)):
        if S[j] < S[menor]:
            menor = j
    return menor
```

Se guarda el índice (no el valor) y se actualiza solo con `<` estricto, de modo que ante un empate se conserva el primero.
</details>

### Ejercicio 5 — Selección con la función auxiliar

Escribe `seleccion(lista)` que devuelva una lista nueva ordenada usando selección in situ sobre una copia. Reutiliza `posicion_minimo` (ejercicio 4; si no lo resolviste, ejecuta primero la solución).

In [28]:
def seleccion(lista):
    # Escribe tu solución aquí
    pass

In [29]:
verificar("lista del PDF", seleccion([50, 30, 40, 10, 20]), [10, 20, 30, 40, 50])
verificar("vacía", seleccion([]), [])
verificar("con repetidos", seleccion([3, 1, 3, 2]), [1, 2, 3, 3])

Incorrecto: lista del PDF. Se esperaba [10, 20, 30, 40, 50] y se obtuvo None
Incorrecto: vacía. Se esperaba [] y se obtuvo None
Incorrecto: con repetidos. Se esperaba [1, 2, 3, 3] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def seleccion(lista):
    S = list(lista)
    for i in range(len(S) - 1):
        menor = posicion_minimo(S, i)
        S[i], S[menor] = S[menor], S[i]
    return S
```

Para cada posición `i` se busca el índice del menor del resto y se hace un único intercambio.
</details>

### Ejercicio 6 — Contar las comparaciones de inserción

Escribe `comparaciones_insercion(lista)` que cuente cuántas veces se evalúa `S[j] > x` en `insertionsort2` (pregunta Q2 de la prueba sorpresa). Recuerda que si `j < 0` no se evalúa.

In [30]:
def comparaciones_insercion(lista):
    # Escribe tu solución aquí
    pass

In [31]:
verificar("lista del PDF", comparaciones_insercion([50, 30, 40, 10, 20]), 10)
verificar("mejor caso (ordenada)", comparaciones_insercion([1, 2, 3, 4]), 3)
verificar("peor caso (inversa)", comparaciones_insercion([4, 3, 2, 1]), 6)

Incorrecto: lista del PDF. Se esperaba 10 y se obtuvo None
Incorrecto: mejor caso (ordenada). Se esperaba 3 y se obtuvo None
Incorrecto: peor caso (inversa). Se esperaba 6 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def comparaciones_insercion(lista):
    S = list(lista)
    cuenta = 0
    for i in range(1, len(S)):
        x = S[i]
        j = i - 1
        while j >= 0:
            cuenta += 1
            if S[j] > x:
                S[j + 1] = S[j]
                j -= 1
            else:
                break
        S[j + 1] = x
    return cuenta
```

Cada vez que `j >= 0` se evalúa `S[j] > x`, por eso se suma 1 en cada vuelta del `while`. Ordenada: `N-1`; inversa: `N(N-1)/2`.
</details>

### Ejercicio 7 — Burbuja con corte anticipado

Escribe `pasadas_burbuja(lista)` que ordene una copia con burbuja optimizada (con bandera) y devuelva **cuántas pasadas** hizo hasta terminar, contando la pasada que no intercambia nada (si existe). El bucle externo hace como máximo `N - 1` pasadas.

In [32]:
def pasadas_burbuja(lista):
    # Escribe tu solución aquí
    pass

In [33]:
verificar("ya ordenada", pasadas_burbuja([1, 2, 3]), 1)
verificar("inversa de 3 (máximo N-1 pasadas)", pasadas_burbuja([3, 2, 1]), 2)
verificar("casi ordenada", pasadas_burbuja([2, 1, 3, 4, 5]), 2)

Incorrecto: ya ordenada. Se esperaba 1 y se obtuvo None
Incorrecto: inversa de 3 (máximo N-1 pasadas). Se esperaba 2 y se obtuvo None
Incorrecto: casi ordenada. Se esperaba 2 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def pasadas_burbuja(lista):
    S = list(lista)
    n = len(S)
    pasadas = 0
    for i in range(n - 1):
        pasadas += 1
        hubo_intercambio = False
        for j in range(n - 1 - i):
            if S[j] > S[j + 1]:
                S[j], S[j + 1] = S[j + 1], S[j]
                hubo_intercambio = True
        if not hubo_intercambio:
            break
    return pasadas
```

Se cuenta cada pasada y se corta en la primera que no intercambia nada. En una lista ordenada basta una pasada; eso es lo que hace a burbuja optimizada `O(N)` en el mejor caso.
</details>

### Ejercicio 8 — Inserción estable por longitud

Escribe `ordenar_por_longitud(palabras)` que devuelva una lista nueva con las palabras ordenadas por longitud (menor a mayor) usando inserción. Si dos palabras miden lo mismo deben conservar su orden original (ordenamiento **estable**).

In [34]:
def ordenar_por_longitud(palabras):
    # Escribe tu solución aquí
    pass

In [35]:
verificar("estable", ordenar_por_longitud(["pera", "uva", "kiwi", "fig"]), ["uva", "fig", "pera", "kiwi"])
verificar("ya ordenada", ordenar_por_longitud(["a", "bb", "ccc"]), ["a", "bb", "ccc"])
verificar("vacía", ordenar_por_longitud([]), [])

Incorrecto: estable. Se esperaba ['uva', 'fig', 'pera', 'kiwi'] y se obtuvo None
Incorrecto: ya ordenada. Se esperaba ['a', 'bb', 'ccc'] y se obtuvo None
Incorrecto: vacía. Se esperaba [] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def ordenar_por_longitud(palabras):
    S = list(palabras)
    for i in range(1, len(S)):
        x = S[i]
        j = i - 1
        while j >= 0 and len(S[j]) > len(x):
            S[j + 1] = S[j]
            j -= 1
        S[j + 1] = x
    return S
```

La condición usa `>` estricto: los elementos de igual longitud **no** se desplazan, así que conservan su orden relativo. Con `>=` la inserción dejaría de ser estable.
</details>

## Apuntes para estudiar

**Clave**

- Ordenar = reubicar elementos usando comparaciones e intercambios; `S[x], S[y] = S[y], S[x]` es el intercambio idiomático.
- Burbuja: compara adyacentes e intercambia si están invertidos; tras cada pasada el mayor queda al final.
- Selección: busca el menor de la parte no ordenada y lo lleva a su posición; hace pocos intercambios.
- Inserción: toma cada elemento y lo inserta en su lugar dentro de la parte ordenada, desplazando los mayores.
- Burbuja y selección: `O(N²)` siempre (con el código del PDF). Inserción: `O(N)` en el mejor caso y `O(N²)` en el peor.
- Comparaciones totales de un doble bucle triangular: `N(N-1)/2`.

**Errores comunes**

- Intercambiar sin variable temporal ni asignación múltiple (se pierde un valor).
- Recorrer `range(n)` en el bucle interno de burbuja y salirse del índice con `S[j + 1]`.
- Guardar el valor del mínimo en vez de su índice en selección.
- Escribir `S[j] > x and j >= 0` en inserción: `S[-1]` no falla pero compara mal.
- Olvidar que `selectionsort1` e `insertionsort1` vacían la lista original (`pop`).
- Suponer que el código de burbuja del PDF hace `N(N-1)/2` comparaciones: hace `N(N-1)`.

**Para el quiz**

- Los tres algoritmos del tema son de tiempo cuadrático `O(N²)` (palabra clave: "tiempo cuadrático").
- El mejor caso de inserción es la lista ya ordenada: `O(N)`; el peor, la lista inversa.
- Los intercambios de burbuja equivalen al número de inversiones: para `[50, 30, 40, 10, 20]` son 8.
- Comparaciones de inserción para esa misma lista: 10.
- Versión con lista extra frente a in situ: la primera usa memoria adicional `R`; la segunda ordena en la misma lista.

## Autoevaluación

**Pregunta 1.** ¿Qué diferencia fundamental hay entre la estrategia de burbuja y la de selección?

<details><summary>Ver respuesta</summary>

Burbuja compara e intercambia solo elementos **adyacentes** y va llevando el mayor hacia el final; selección busca el **menor** de la parte no ordenada (que puede estar lejos) y lo intercambia directamente con la primera posición no ordenada, con pocos intercambios.
</details>

**Pregunta 2.** ¿Por qué el PDF dice que el ordenamiento de burbujas tiene complejidad `O(N²)`?

<details><summary>Ver respuesta</summary>

Porque ejecuta dos bucles `for` anidados: el número de comparaciones por pasada es `N-1, N-2, ..., 1`, cuya suma es `N(N-1)/2`, un polinomio de grado 2 en `N`.
</details>

**Pregunta 3.** ¿Cuándo es `O(N)` el ordenamiento por inserción y cuándo `O(N²)`?

<details><summary>Ver respuesta</summary>

Es `O(N)` en el mejor caso (lista ya ordenada), porque cada elemento se compara una sola vez con su vecino. Es `O(N²)` en el peor caso (lista en orden inverso), porque cada elemento se compara con todos los de la parte ordenada.
</details>

**Pregunta 4.** ¿Qué ventaja tiene la versión in situ frente a la que usa una lista extra `R`?

<details><summary>Ver respuesta</summary>

No usa memoria adicional: ordena directamente en la misma lista. Además no vacía la lista original por `pop`, sino que la modifica ordenándola.
</details>

**Pregunta 5.** ¿Por qué `S[x], S[y] = S[y], S[x]` intercambia correctamente sin variable temporal?

<details><summary>Ver respuesta</summary>

Porque Python evalúa primero el lado derecho completo como una tupla `(S[y], S[x])` y solo después asigna sus valores a las posiciones de la izquierda, por lo que ningún valor se pisa antes de usarse.
</details>

---

[Índice de la semana](../semana-04.ipynb) | [Anterior](../semana-04.ipynb) | [Siguiente](u28-ordenamiento-por-fusion.ipynb)